# 第12章 目标检测

本章节学习目标：

- 理解本章核心算法的**数学原理**
- 掌握算法的**手写实现**方法
- 学会使用 OpenCV 对应函数进行**工程实践**
- 通过编程练习加深对算法的理解

> **📌 学习建议**：先阅读概念说明，再动手编写代码，最后完成练习


In [ ]:
# -*- coding: utf-8 -*-
# 中文路径兼容的图像读写函数
import numpy as np
import cv2
import os

def cv_imread(filepath, flags=cv2.IMREAD_COLOR):
    """支持中文路径的图像读取"""
    with open(filepath, 'rb') as f:
        buf = np.frombuffer(f.read(), dtype=np.uint8)
    return cv2.imdecode(buf, flags)

def cv_imwrite(filepath, img):
    """支持中文路径的图像写入"""
    ext = os.path.splitext(filepath)[1]
    success, buf = cv2.imencode(ext, img)
    if success:
        with open(filepath, 'wb') as f:
            f.write(buf.tobytes())
        return True
    return False


# 代码实现

由于Faster R-CNN代码架构过于庞大，我们将直接调用相关接口进行效果展示。先导入必要的包，并使用在MS COCO数据集上预训练的ResNet50作为主干网络。

In [ ]:
import os
import numpy as np
import functools
import matplotlib.pyplot as plt
import cv2
import torch
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

In [ ]:
! pip install -U 'git+https://github.com/MS COCOdataset/MS COCOapi.git#subdirectory=PythonAPI'
! git clone https://github.com/pytorch/vision.git
! cd vision;cp references/detection/utils.py ../
! cp references/detection/transforms.py ../
! cp references/detection/MS COCO_eval.py ../
! cp references/detection/engine.py ../
! cp references/detection/MS COCO_utils.py ../

In [ ]:
# 加载模型，使用MS COCO数据集预训练
model = torchvision.models.detection.fasterrcnn_resnet50_fpn(pretrained='MS COCO')
num_classes = 21  
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

# 初始化优化器与学习率
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.001, weight_decay=0.0005)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.1)

In [ ]:
WEIGHTS_FILE = "../input/fasterrcnn/faster_rcnn_state.pth"
model.load_state_dict(torch.load(WEIGHTS_FILE))

接着，编写模型测试的代码。

In [ ]:
# 对模型进行测试
def obj_detector(img):
    img = cv2.imread(img, cv2.IMREAD_COLOR)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB).astype(np.float32)

    # 导入图像并对图像进行处理
    img /= 255.0
    img = torch.from_numpy(img)
    img = img.unsqueeze(0)
    img = img.permute(0,3,1,2)
    
    model.eval()
    
    # 设置阈值
    detection_threshold = 0.70
    
    img = list(im.to(device) for im in img)
    output = model(img)

    for i , im in enumerate(img):
        boxes = output[i]['boxes'].data.cpu().numpy()
        scores = output[i]['scores'].data.cpu().numpy()
        labels = output[i]['labels'].data.cpu().numpy()
        
        labels = labels[scores >= detection_threshold]
        boxes = boxes[scores >= detection_threshold].astype(np.int32)
        scores = scores[scores >= detection_threshold]

        boxes[:, 2] = boxes[:, 2] - boxes[:, 0]
        boxes[:, 3] = boxes[:, 3] - boxes[:, 1]
    
    sample = img[0].permute(1,2,0).cpu().numpy()
    sample = np.array(sample)
    
    boxes = output[0]['boxes'].data.cpu().numpy()
    name = output[0]['labels'].data.cpu().numpy()
    scores = output[0]['scores'].data.cpu().numpy()
    
    boxes = boxes[scores >= detection_threshold].astype(np.int32)
    names = name.tolist()
    
    return names, boxes, sample

在ImageNet上测试模型的效果。

In [ ]:
pred_path = "../input/imagenet/imagenet/val/"
pred_files = [os.path.join(pred_path,f) for f in os.listdir(pred_path)]

classes= {1:'aeroplane', 2:'bicycle', 3:'bird', 4:'boat', 5:'bottle',
          6:'bus', 7:'car', 8:'cat', 9:'chair', 10:'cow',
          11:'diningtable', 12:'dog', 13:'horse', 14:'motorbike',
          15:'person', 16:'pottedplant', 17:'sheep', 18:'sofa', 
          19:'train',20:'tvmonitor'}

plt.figure(figsize=(20, 60))
image_list = [0,11,17,28]
for i, images in enumerate(pred_files):
    if i > 30:
        break
    if i not in image_list:
        continue

    plt.subplot(10,2,image_list.index(i)+1)
    
    names, boxes, sample = obj_detector(images)
    
    for i,box in enumerate(boxes):
        # 绘制包围盒
        cv2.rectangle(sample, (box[0], box[1]), (box[2], box[3]), (0, 220, 0), 2)
        cv2.putText(sample, classes[names[i]], (box[0],box[1]-5), cv2.FONT_HERSHEY_COMPLEX,
                    0.7, (220,0,0), 1, cv2.LINE_AA)  

    plt.axis('off')
    plt.imshow(sample)



---

## 📝 练习：本章算法手写实现与扩展


**练习目标**：基于本章所学内容，完成以下实践任务。

**要求**：
1. 手写实现本章的核心算法（不直接调用 OpenCV/PyTorch 对应函数）
2. 使用本章学习的方法处理至少 2 张不同的测试图像
3. 对比手写实现与现成库函数的结果差异
4. 分析算法参数对结果的影响
5. 撰写 200 字以上的实验报告


**💡 小提示**：
- 除 `cv_imread` / `cv_imwrite` 外，不直接调用 OpenCV 高层函数
- 使用 NumPy 进行矩阵运算
- 注意边界处理和数值范围
- 对比手写实现与库函数的结果



<details>
<summary><b>🔑 点击查看完整解决方案</b></summary>

---

### 解决方案详解



In [ ]:
```python
# 本章练习代码框架
import numpy as np
import cv2
import os
import matplotlib.pyplot as plt

def cv_imread(filepath, flags=cv2.IMREAD_COLOR):
    """支持中文路径的图像读取"""
    with open(filepath, 'rb') as f:
        buf = np.frombuffer(f.read(), dtype=np.uint8)
    return cv2.imdecode(buf, flags)

def cv_imwrite(filepath, img):
    """支持中文路径的图像写入"""
    ext = os.path.splitext(filepath)[1]
    success, buf = cv2.imencode(ext, img)
    if success:
        with open(filepath, 'wb') as f:
            f.write(buf.tobytes())
        return True
    return False

# ============================================
# TODO: 在此处手写实现本章核心算法
# ============================================

# 示例框架：
# 1. 数据准备
# img = cv_imread('test_image.jpg')
# gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# 2. 手写算法实现
# def algorithm_manual(input_image, **params):
#     # TODO: 实现算法核心逻辑
#     # 要求：除 OpenCV 读写函数外，其余代码手写
#     return output

# 3. 对比验证
# result_manual = algorithm_manual(gray)
# result_library = cv2.XXX(gray)  # 对应库函数
# diff = np.abs(result_manual.astype(float) - result_library.astype(float))
# print(f"最大差异: {diff.max()}")

# 4. 参数敏感性分析
# for param in [param1, param2, param3]:
#     result = algorithm_manual(gray, param=param)
#     # 可视化结果变化

# 5. 实验报告
print("请完成上述练习并撰写实验报告")
```



### 💻 代码要点解释

1. **图像读取与保存**：使用自定义的 `cv_imread` / `cv_imwrite` 函数，解决 Windows 中文路径下 OpenCV 读写图像失败的问题

2. **算法核心**：手写实现的核心在于**不依赖现成库函数**，而是直接操作像素和矩阵运算

3. **对比验证**：通过与 OpenCV 对应函数的结果进行数值对比，验证手写实现的正确性

4. **参数分析**：调整算法参数，观察输出变化，理解每个参数的物理含义

5. **扩展思考**：尝试将算法应用到自己的图像上，或改进算法（如增加加速技巧）

---

</details>

---
